# 01 — Find and Catalogue

Runs the full matchup discovery pipeline for a single Sentinel-2A / Landsat-9 crossover
event on 2022-05-21 and saves the result as a local STAC catalogue.

**What happens:**
1. `find_and_catalogue` reads the orbit-crossover NetCDF from `find_and_catalogue.yaml`
   and calls `orbitx` to identify candidate events in the requested time window.
2. For each event, `scrappi` is queried for real products via EODAG.
3. Products with overlapping footprints are paired into `Matchup` objects.
4. Everything is serialised as STAC Items inside a local `pystac` catalogue.

The catalogue is written to `example/catalogue/` (relative to the repo root) — subsequent
notebooks open that same directory.

**Prerequisites:** `eomatch` installed in your environment and
`find_and_catalogue.yaml` present alongside this notebook.

In [ ]:
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

# Resolve paths relative to this notebook's location so the notebook works
# regardless of the working directory Jupyter was launched from.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
CONFIG_PATH = os.path.join(NOTEBOOK_DIR, "find_and_catalogue.yaml")
CATALOGUE_DIR = os.path.join(NOTEBOOK_DIR, "catalogue")

print("Config:   ", CONFIG_PATH)
print("Catalogue:", CATALOGUE_DIR)

## Run the pipeline

`find_and_catalogue` is idempotent — re-running it refreshes the catalogue in place.

In [ ]:
from eomatch.context import EOMatchContext
from eomatch.find_and_catalogue import find_and_catalogue

context = EOMatchContext(CONFIG_PATH)

catalogue = find_and_catalogue(context=context, path=CATALOGUE_DIR)

## Inspect the result

Walk the in-memory catalogue to see which collections and items were created.

In [ ]:
print(f"Catalogue root: {catalogue.path}")
print()

for collection in catalogue.catalog.get_children():
    items = list(collection.get_items())
    print(f"  [{collection.id}]  ({len(items)} item(s))")
    for item in items:
        print(f"    {item.id}")

## Event and matchup summary

In [ ]:
events = catalogue.get_events()
print(f"Events: {len(events)}")

for event in events:
    mu_set = event.matchup_set
    n = len(mu_set) if mu_set is not None else 0
    print(f"\n  {event.stac_id}")
    print(f"  Platforms:   {event.platforms}")
    print(f"  Collections: {event.collections}")
    print(f"  Window:      {event.start_time} – {event.stop_time}")
    print(f"  Matchups:    {n}")

    if mu_set is not None:
        for i, mu in enumerate(mu_set):
            print(f"    [{i}] collocation bounds: {mu.collocation_region.bounds}")
            print(f"         time diff (s):       {mu.time_diff_abs:.1f}")
            for p in mu.products:
                print(f"         {p.collection}: {p.id}")

## On-disk layout

The catalogue directory mirrors the STAC collection hierarchy.

In [ ]:
for root, dirs, files in os.walk(CATALOGUE_DIR):
    dirs.sort()
    depth = root.replace(CATALOGUE_DIR, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    for fname in sorted(files):
        print(f"{indent}  {fname}")